# Sección 1 · Del VCF a una matriz limpia

*Importación y control de calidad de datos SNP — genética de poblaciones en* Nothofagus pumilio *(lenga)*

> **Cómo usar este notebook:** ejecuta las celdas en orden (Shift+Enter). Las celdas de ejercicio (E1, E2…) están casi vacías para que las completes. Al final está la **Tarea**, que revisaremos al comienzo de la próxima sección.


In [ ]:
# ---- Preparación (ejecutar primero) ----
# Tamaño de gráficos en Jupyter
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 110)

source("R/helpers.R")
if (!file.exists("data/lenga_sim.vcf")) source("00_simular_datos.R")

## Objetivos de aprendizaje

Al terminar esta sección sabrás:

1. **Leer** un archivo VCF y convertirlo en una matriz de dosis alélicas (0/1/2/NA).
2. **Diagnosticar** la calidad de tus datos: datos faltantes por individuo y por SNP, frecuencia del alelo menor (MAF) y ligamiento (LD).
3. **Decidir y justificar** umbrales de filtrado, y dejarlos documentados para tu sección de Métodos.

> **Sobre los datos.** Trabajamos con un VCF *simulado* (`00_simular_datos.R`): 20 poblaciones × 12 árboles × 1.500 SNP a lo largo de ~2.000 km de latitud. Es un andamio para practicar el flujo con una «verdad» conocida; **no son datos biológicos reales**. El mismo código se aplica después a tus datos reales.


## Conceptos previos

- **SNP** (*Single Nucleotide Polymorphism*) — posición del genoma en la que individuos distintos llevan nucleótidos diferentes. Es el marcador estándar de la genómica de poblaciones.
- **Genotipo vs. dosis alélica.** El genotipo describe qué alelos lleva un individuo en un locus (p. ej. `A/A`, `A/G`, `G/G`). La **dosis** es un número que cuenta cuántas copias del alelo alternativo (ALT) lleva: `0`, `1` o `2`. Es la representación con la que trabajaremos en casi todo el curso.
- **VCF** (*Variant Call Format*) — formato estándar para almacenar variantes. En un proyecto real el VCF sale de un *pipeline* (mapeo → llamado de variantes → filtros). Ese pipeline ya aplicó un primer filtro, **pero no es el filtro de tu pregunta biológica**.
- **Control de calidad (QC)** — diagnosticar y filtrar datos para que tus conclusiones no sean artefactos. Hoy revisamos tres cantidades: **datos faltantes**, **MAF** y **LD**.

**Pregunta para anticipar:** si un individuo tiene el 40 % de sus genotipos faltantes, ¿qué crees que le pasa a su heterocigosidad? *(La respondemos al final de la sección.)*


## 1. Qué hay dentro de un VCF

Cada fila es un SNP; cada columna (desde `FORMAT`) es un individuo. El campo `GT` es el genotipo de cada individuo:

- `0/0` → homocigoto del alelo de referencia (REF)
- `0/1` → heterocigoto (una copia REF y una ALT)
- `1/1` → homocigoto del alelo alternativo (ALT)
- `./.` → genotipo **faltante** (no se pudo llamar)

Miremos las primeras líneas del archivo:


In [ ]:
lineas <- readLines("data/lenga_sim.vcf", n = 20)
cat(substr(lineas[1:3], 1, 90), sep = "\n")
cat(substr(lineas[length(lineas)], 1, 120), "...\n")

Las líneas que empiezan con `##` son **metadatos** (formato, contigs). La línea `#CHROM` es la cabecera de columnas. A partir de ahí, cada fila es un SNP con su posición (`POS`), sus alelos (`REF`/`ALT`) y el genotipo de cada individuo.


## 2. Importar y convertir a dosis

El primer paso es llevar el VCF a una matriz **individuos × SNP** donde cada celda es la dosis (`0`, `1`, `2` o `NA`). `read_vcf_dosage()` (en `R/helpers.R`) hace exactamente eso con `vcfR`.

**Regla de oro:** cada fila de `G` debe corresponder a la *misma* fila de `meta`. El `stopifnot()` de abajo lo verifica; si alguna vez lo quitáramos, todo se calcularía «bien» y todo estaría mal, sin avisar.


In [ ]:
x <- read_vcf_dosage("data/lenga_sim.vcf")
G <- x$G
dim(G)            # individuos x SNP
G[1:5, 1:8]

meta <- read.csv("data/lenga_sim_muestras.csv", stringsAsFactors = FALSE)
meta <- meta[match(rownames(G), meta$id), ]
stopifnot(all(rownames(G) == meta$id))      # ¡siempre verificar el orden!
head(meta)

## 3. Dónde están las muestras

Antes de analizar conviene **ver** el diseño: ¿dónde está cada población? Aquí hay 20 sitios a lo largo de ~2.000 km de latitud, con muestreo parejo (12 individuos por sitio).

> **Pregunta:** ¿qué sesgo introduciría un muestreo muy desigual entre sitios? *(Guárdala: reaparece en la Sección 2, con los alelos privados.)*


In [ ]:
pops <- aggregate(cbind(lat, lon, alt) ~ pop, data = meta, FUN = mean)
pops$n <- as.integer(table(meta$pop)[pops$pop])
ggplot(pops, aes(lon, lat, size = n, colour = alt)) +
  geom_point() + geom_text(aes(label = pop), nudge_x = 0.35, size = 3, colour = "grey30", show.legend = FALSE) +
  scale_colour_viridis_c() + labs(x = "Longitud", y = "Latitud", colour = "Altitud (m)", size = "n") +
  coord_fixed(1.2)

## 4. Diagnóstico de datos faltantes

El **call rate** es la proporción de genotipos llamados (lo opuesto al dato faltante). Se calcula por individuo y por SNP:

- Por **individuo**: muchos faltantes suelen ser señal de muestra de mala calidad (poco ADN, degradación).
- Por **SNP**: muchos faltantes suelen ser regiones difíciles de genotipar (repeticiones, baja cobertura).

**¿Qué esperas?** Mira los histogramas antes de decidir dónde poner el umbral. No hay un número «correcto»: hay un número *justificado*.


In [ ]:
ind_cr <- rowMeans(!is.na(G))                 # call rate por individuo
snp    <- snp_summary(G)

ggplot(data.frame(ind_cr), aes(ind_cr)) + geom_histogram(bins = 30, fill = "steelblue") +
  geom_vline(xintercept = 0.8, lty = 2, colour = "red") +
  labs(x = "Proporción de genotipos llamados (individuo)", y = "N° de individuos")

ggplot(snp, aes(call_rate)) + geom_histogram(bins = 30, fill = "darkorange") +
  geom_vline(xintercept = 0.9, lty = 2, colour = "red") +
  labs(x = "Proporción de genotipos llamados (SNP)", y = "N° de SNP")

head(sort(ind_cr), 8)      # los peores individuos

## 5. Frecuencia del alelo menor (MAF)

La **MAF** es la frecuencia del alelo menos común en la muestra. Va de 0 (SNP **monomórfico**, sin variación) a 0.5 (ambos alelos igual de frecuentes).

Los SNP monomórficos no aportan información sobre *variación*, pero **conviene contarlos y reportarlos**: son parte de la descripción de tus datos.

> **Pregunta:** ¿qué SNP crees que aportan más información sobre estructura poblacional: los raros o los comunes? *(Lo veremos en la Sección 3.)*


In [ ]:
ggplot(snp, aes(maf)) + geom_histogram(bins = 40, fill = "seagreen") +
  geom_vline(xintercept = 0.05, lty = 2, colour = "red") +
  labs(x = "MAF", y = "N° de SNP")
sum(snp$maf == 0, na.rm = TRUE)    # SNPs monomórficos

## 6. Filtrado y guardado

`qc_filter()` aplica tres umbrales: call rate por individuo, call rate por SNP y MAF mínima. Fíjate en cuánto se pierde en cada etapa.

**Idea clave:** los filtros son **decisiones científicas con consecuencias**, no un trámite. Por eso se escriben en Métodos. En particular, el filtro de MAF cambia las estimaciones de diversidad (lo veremos en la Sección 2 como *sesgo de verificación*).


In [ ]:
q <- qc_filter(G, ind_cr = 0.80, snp_cr = 0.90, maf = 0.05)
G_clean    <- q$G
meta_clean <- meta[q$keep_ind, ]

resumen <- data.frame(
  etapa = c("Original", "Filtrado"),
  individuos = c(nrow(G), nrow(G_clean)),
  snps = c(ncol(G), ncol(G_clean)),
  faltante_pct = round(100 * c(mean(is.na(G)), mean(is.na(G_clean))), 2))
resumen

saveRDS(G_clean, "data/geno_clean.rds")
saveRDS(meta_clean, "data/meta_clean.rds")
table(meta_clean$pop)

## 7. Ligamiento (LD): ¿son independientes mis SNP?

Muchos métodos asumen SNP aproximadamente independientes. El **desequilibrio de ligamiento (LD)** mide cuánto se heredan juntos dos SNP; el `r²` entre pares es una medida común (0 = independientes, 1 = información redundante).

Aquí lo diagnosticamos con el `r²` entre SNP **vecinos**. En la simulación los SNP son independientes, así que esperamos `r² ≈ 1/n` (muy bajo). En lenga real esperaríamos más ligamiento entre SNP cercanos dentro del mismo contig, y habría que **podar por LD** (p. ej. `SNPRelate::snpgdsLDpruning` o PLINK `--indep-pairwise`).


In [ ]:
Gi  <- impute_mean(G_clean)
adj <- suppressWarnings(sapply(seq_len(ncol(Gi) - 1), function(j) cor(Gi[, j], Gi[, j + 1])^2))
mean(adj, na.rm = TRUE)    # r2 medio entre SNP vecinos

---

## Ejercicios

**E1.** ¿Cuántos individuos y SNP se pierden con `ind_cr = 0.80`? Identifica los individuos descartados y verifica si pertenecen a una misma población.

> **Idea clave:** si la pérdida se concentra en una población, el filtro introduce sesgo de muestreo.


In [ ]:
# TU CÓDIGO AQUÍ

**E2.** Construye una tabla de sensibilidad: número de SNP retenidos para `snp_cr` en {0.8, 0.9, 0.95} y `maf` en {0.01, 0.05, 0.10}. ¿Qué combinación elegirías y por qué?

> **Idea clave:** no hay umbral «correcto»; hay umbrales *justificados y reportados*.


In [ ]:
# Pista: expand.grid() + sapply() sobre qc_filter()

**E3.** ¿Importa el **orden** de los filtros (primero individuos o primero SNP)? Reescribe `qc_filter` para filtrar primero los SNP y compara los resultados.

> **Idea clave:** el orden altera qué individuos se descartan; documéntalo.


In [ ]:
# TU CÓDIGO AQUÍ

**E4.** Calcula la heterocigosidad observada por SNP y marca los SNP con Ho > 0.6. En lenga real, un exceso de heterocigotos suele indicar *paralogía* (lecturas de copias distintas mapeando juntas). ¿Los eliminarías?

> **Idea clave:** en la simulación probablemente no haya exceso de heterocigotos, y **eso también es un resultado**.


In [ ]:
# TU CÓDIGO AQUÍ

## Errores y dudas frecuentes

- **«Me sale `cannot open file 'R/helpers.R'`»** → Jupyter se abrió desde otra carpeta. Ciérralo y ejecútalo dentro de la carpeta del curso.
- **«El kernel no es R»** → cámbialo arriba a la derecha (debe decir **R**).
- **«¿Por qué quito monomórficos si son datos?»** → no aportan información sobre variación; pero cuéntalos y repórtalos.
- **«¿Y los SNP ligados?»** → en datos reales se podan por LD para estructura; para diversidad se discute.


## Resumen: ideas que debes llevarte

1. Un VCF es una tabla de variantes; lo convertimos a una **matriz de dosis** (individuos × SNP, 0/1/2/NA).
2. `G` y `meta` deben estar **alineados por fila**; verifícalo siempre.
3. El QC revisa **faltantes, MAF y LD**; cada umbral es una decisión con consecuencias.
4. Lo que decidas hoy condiciona todo lo que viene: **escríbelo en Métodos**.

> *Respuesta a la pregunta de anticipación:* un individuo con mucho dato faltante suele tener la heterocigosidad **sesgada a la baja** (los heterocigotos se llaman peor con poca cobertura), lo que distorsiona cualquier comparación entre poblaciones.


## Tarea (se revisa al comienzo de la próxima sección)

Escribe en **5 líneas** los umbrales de QC que usarás y su justificación. Será la sección *Métodos* de tu reporte.

*Pregunta de reflexión:* tu proyecto trata de diversidad y estructura. ¿Qué sesgos podría introducir cada filtro (MAF, call rate, LD) sobre esas dos cantidades?
